# 🧠 ANN Project — Full Walkthrough
Breast-cancer classification with a neural network built from scratch in NumPy.

**Sections:** 1) Setup · 2) Data · 3) Model · 4) Training · 5) Evaluation · 6) Save/Load · 7) Compare with scikit-learn

## 1. Setup

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # so `src` is importable
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.config import load_config
from src.data import prepare_data, load_raw
from src.model import ANN
from src.metrics import classification_report
from src.utils import set_seed, plot_history, plot_confusion_matrix

cfg = load_config()
set_seed(cfg['seed'])
cfg

{'seed': 42,
 'data': {'test_size': 0.2, 'val_size': 0.15},
 'model': {'hidden_layers': [32, 16],
  'hidden_activation': 'relu',
  'dropout': 0.2,
  'l2': 0.0005},
 'training': {'optimizer': 'adam',
  'learning_rate': 0.001,
  'epochs': 300,
  'batch_size': 32,
  'patience': 25},
 'paths': {'raw_data': 'C:\\Users\\Dr. Shubham Khairnar\\Desktop\\New folder\\ann-project\\ann-project\\data\\raw\\breast_cancer.csv',
  'processed_dir': 'C:\\Users\\Dr. Shubham Khairnar\\Desktop\\New folder\\ann-project\\ann-project\\data\\processed',
  'model_path': 'C:\\Users\\Dr. Shubham Khairnar\\Desktop\\New folder\\ann-project\\ann-project\\models\\ann_model.npz',
  'figures_dir': 'C:\\Users\\Dr. Shubham Khairnar\\Desktop\\New folder\\ann-project\\ann-project\\reports\\figures'}}

## 2. Data exploration

In [2]:
df = load_raw(cfg['paths']['raw_data'])
print(df.shape)
df.head()

(569, 31)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [3]:
df['target'].value_counts().rename({0:'malignant',1:'benign'}).plot(kind='bar', title='Class balance')
plt.show()

C:\Users\Dr. Shubham Khairnar\AppData\Local\Temp\ipykernel_28000\2230384411.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
d = prepare_data(cfg)
{k: v.shape for k, v in d.items()}

{'X_train': (386, 30),
 'y_train': (386, 1),
 'X_val': (69, 30),
 'y_val': (69, 1),
 'X_test': (114, 30),
 'y_test': (114, 1)}

## 3. Build the model

In [5]:
m = cfg['model']
model = ANN(d['X_train'].shape[1], m['hidden_layers'], m['hidden_activation'], m['dropout'], m['l2'])
for i, l in enumerate(model.layers):
    print(f'Layer {i}: {l.W.shape[0]} -> {l.W.shape[1]}  ({l.activation_name})')

Layer 0: 30 -> 32  (relu)
Layer 1: 32 -> 16  (relu)
Layer 2: 16 -> 1  (sigmoid)


## 4. Train

In [6]:
t = cfg['training']
hist = model.fit(d['X_train'], d['y_train'], d['X_val'], d['y_val'],
                 epochs=t['epochs'], batch_size=t['batch_size'], optimizer=t['optimizer'],
                 lr=t['learning_rate'], patience=t['patience'])

Epoch    1 | loss 0.6326 | val_loss 0.6203 | acc 0.7358 | val_acc 0.7536
Epoch   10 | loss 0.1864 | val_loss 0.2031 | acc 0.9560 | val_acc 0.9420
Epoch   20 | loss 0.1073 | val_loss 0.1305 | acc 0.9767 | val_acc 0.9710
Epoch   30 | loss 0.0835 | val_loss 0.1148 | acc 0.9845 | val_acc 0.9855
Epoch   40 | loss 0.0705 | val_loss 0.1082 | acc 0.9845 | val_acc 0.9855
Epoch   50 | loss 0.0624 | val_loss 0.1087 | acc 0.9922 | val_acc 0.9855
Epoch   60 | loss 0.0553 | val_loss 0.1079 | acc 0.9922 | val_acc 0.9710
Epoch   70 | loss 0.0503 | val_loss 0.1050 | acc 0.9922 | val_acc 0.9710
Epoch   80 | loss 0.0460 | val_loss 0.1038 | acc 0.9922 | val_acc 0.9710
Epoch   90 | loss 0.0424 | val_loss 0.1097 | acc 0.9922 | val_acc 0.9710
Epoch  100 | loss 0.0395 | val_loss 0.1091 | acc 0.9948 | val_acc 0.9710
Early stopping at epoch 101


In [7]:
plot_history(hist, f"{cfg['paths']['figures_dir']}/training_curves.png")
plt.show()

C:\Users\Dr. Shubham Khairnar\AppData\Local\Temp\ipykernel_28000\2115214904.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Evaluate

In [8]:
rep = classification_report(d['y_test'], model.predict(d['X_test']))
pd.Series({k: v for k, v in rep.items() if k != 'confusion_matrix'}).round(4)

accuracy     0.9649
precision    0.9857
recall       0.9583
f1           0.9718
dtype: float64

In [9]:
plot_confusion_matrix(rep['confusion_matrix'], save_path=f"{cfg['paths']['figures_dir']}/confusion_matrix.png")
plt.show()

C:\Users\Dr. Shubham Khairnar\AppData\Local\Temp\ipykernel_28000\868352617.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Save & reload

In [10]:
model.save(cfg['paths']['model_path'])
reloaded = ANN(d['X_test'].shape[1], m['hidden_layers'], m['hidden_activation']).load(cfg['paths']['model_path'])
print('Same predictions after reload:', np.array_equal(model.predict(d['X_test']), reloaded.predict(d['X_test'])))

Same predictions after reload: True


## 7. Sanity check vs scikit-learn's MLP

In [11]:
from sklearn.neural_network import MLPClassifier
sk = MLPClassifier(hidden_layer_sizes=tuple(m['hidden_layers']), max_iter=500, random_state=cfg['seed'])
sk.fit(d['X_train'], d['y_train'].ravel())
print('sklearn MLP test accuracy :', round(sk.score(d['X_test'], d['y_test'].ravel()), 4))
print('our NumPy ANN test accuracy:', round(rep['accuracy'], 4))

sklearn MLP test accuracy : 0.9561
our NumPy ANN test accuracy: 0.9649


## Next steps
- Try `tanh`/`sigmoid`, more layers, different learning rates
- Add k-fold cross-validation
- Swap in your own CSV in `data/raw/` (needs a `target` column)